In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class LSTM_Params_Config:
    input_size:  int
    hidden_size: int
    num_layers:  int
    input_steps: int
    output_steps: int
    epochs:     int
    batch_size: int
    lr:         float

@dataclass(frozen=True)
class LSTM_Predictor_Config:
    root_dir:       Path
    model_dir:      Path
    lstm_params:    LSTM_Params_Config

In [3]:
from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

class ConfigManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH, 
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
        
    def get_lstm_config(self) -> LSTM_Predictor_Config:
        config = self.config.lstm_config
        params = self.params.lstm_params
        create_directories([config.root_dir])
        
        return LSTM_Predictor_Config(
            root_dir  = Path(config.root_dir),
            model_dir = Path(config.root_dir) / "lstm_model.pth",
            
            lstm_params = LSTM_Params_Config(
                input_size  = params.input_size,
                hidden_size = params.hidden_size,
                num_layers  = params.num_layers,
                input_steps = params.input_steps,
                output_steps = params.output_steps,
                epochs      = params.epochs,
                batch_size  = params.batch_size,
                lr          = params.lr
            ),
        )

In [4]:
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset

from core.logging import logger
from core.exception import CustomException
from lstm.model import LSTM_Predictor
from lstm.utils.data.synthetic_generator import generate_synthetic_metrics

class LSTM_Trainer:
    def __init__(self, config: LSTM_Predictor_Config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = self._get_model()
        
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.config.lstm_params.lr)
        self.criterion = nn.MSELoss()
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, patience=10, factor=0.5, min_lr=1e-6)
        self.loader    = self._prepare_loader()
    
    def _get_model(self):
        return LSTM_Predictor(
            input_size   = self.config.lstm_params.input_size,
            hidden_size  = self.config.lstm_params.hidden_size,
            num_layers   = self.config.lstm_params.num_layers,
            output_steps = self.config.lstm_params.output_steps
        ).to(self.device)
    
    def _prepare_loader(self) -> DataLoader:
        df   = generate_synthetic_metrics()
        data = torch.tensor(df.values, dtype=torch.float32)
        
        self.mean = data.mean(dim=0)
        self.std  = data.std(dim=0)
        data = (data - self.mean) / self.std
        
        X, y = [], []
        inp_steps = self.config.lstm_params.input_steps
        out_steps = self.config.lstm_params.output_steps
        for i in range(len(data) - inp_steps - out_steps):
            X.append(data[i : i + inp_steps])
            y.append(data[i + inp_steps : i + inp_steps + out_steps])
        
        X = torch.stack(X)
        y = torch.stack(y)
        dataset = TensorDataset(X, y)
        
        return DataLoader(dataset, batch_size=self.config.lstm_params.batch_size, shuffle=True)
    
    def train(self):
        epochs = self.config.lstm_params.epochs
        for epoch in range(epochs):
            avg_loss = self._train_one_epoch()
            self.scheduler.step(avg_loss)
            
            if epoch % 10 == 0:
                logger.info(f"Epoch {epoch}/{epochs} — Loss: {avg_loss:.6f} — LR: {self.optimizer.param_groups[0]['lr']:.6f}")
        
        self._save_model()
    
    def _train_one_epoch(self):
        total_loss = 0
        for X_batch, y_batch in self.loader:
            X_batch = X_batch.to(self.device)
            y_batch = y_batch.to(self.device)
                
            self.optimizer.zero_grad()
            y_pred = self.model(X_batch)
            loss   = self.criterion(y_pred, y_batch)
            loss.backward()
            total_loss += loss.item()
            self.optimizer.step()
            
        return total_loss / len(self.loader)
    
    def _save_model(self):
        torch.save({
            "model_state_dict": self.model.state_dict(),
            "mean": self.mean,
            "std":  self.std,
            "input_steps":  self.config.lstm_params.input_steps,
            "output_steps": self.config.lstm_params.output_steps,
        }, self.config.model_dir)
        logger.info(f"LSTM model saved -> {self.config.model_dir}")

In [5]:
import sys

try:
    config = ConfigManager()
    trainer = LSTM_Trainer(config=config.get_lstm_config())
    trainer.train()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-19 17:31:40,833: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-19 17:31:40,838: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-19 17:31:40,840: INFO: common: created directory at: artifacts]
[2026-04-19 17:31:40,841: INFO: common: created directory at: artifacts/lstm]
[2026-04-19 17:31:44,466: INFO: 1262029952: Epoch 0/500 — Loss: 0.961742 — LR: 0.001000]
[2026-04-19 17:31:58,875: INFO: 1262029952: Epoch 10/500 — Loss: 0.930516 — LR: 0.001000]
[2026-04-19 17:32:12,349: INFO: 1262029952: Epoch 20/500 — Loss: 0.545575 — LR: 0.001000]
[2026-04-19 17:32:26,166: INFO: 1262029952: Epoch 30/500 — Loss: 0.292544 — LR: 0.001000]
[2026-04-19 17:32:39,606: INFO: 1262029952: Epoch 40/500 — Loss: 0.079896 — LR: 0.001000]
[2026-04-19 17:32:50,825: INFO: 1262029952: Epoch 50/500 — Loss: 0.039538 — LR: 0.001000]
[2026-04-19 17:33:03,816: INFO: 1262029952: Epoch 60/500 — Loss: 0.027600 — LR: 0.001000]
[2026-04-19 17:33:15,245: INFO: 1262029952: E

In [ ]:
import torch
import httpx
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

from core.logging import logger
from core.exception import CustomException
from lstm.model import LSTM_Predictor

class LSTM_Predictor_:
    def __init__(self, config: LSTM_Predictor_Config,
                 model_path: str = "artifacts/lstm/lstm_model.pth", 
                 prometheus_url: str = "http://localhost:9090"):
        self.config         = config
        self.model_path     = model_path
        self.prometheus_url = prometheus_url
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = self._load_model()
    
    def _load_model(self):
        checkpoint = torch.load(self.model_path, map_location=self.device)
        
        self.input_steps  = checkpoint["input_steps"]
        self.output_steps = checkpoint["output_steps"]
        self.mean = checkpoint["mean"]
        self.std  = checkpoint["std"]
        
        self.model = LSTM_Predictor(
            input_size = self.config.lstm_params.input_size,
            output_steps = self.output_steps
        ).to(self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()
        logger.info("LSTM model loaded")
        
        return self.model
    
    def _query_prometheus(self, query: str) -> list[float]:
        end   = datetime.utcnow()
        start = end - timedelta(minutes=self.input_steps * 15 // 60 + 5)
        
        reponse = httpx.get(
            f"{self.prometheus_url}/api/v1/query_range",
            params={
                "query": query,
                "start": start.isoformat() + "Z",
                "end":   end.isoformat() + "Z",
                "step": "15s"
            }
        )
        result = reponse.json()["data"]["result"]
        if not result:
            return [0.0] * self.input_steps
        
        values = [float(v[1]) for v in result[0]["values"]]
        if len(values) < self.input_steps:
            values = [values[0]] * (self.input_steps - len(values)) + values
        return values[-self.input_steps:] 


    def predict(self) -> dict:
        cpu     = self._query_prometheus('rate(process_cpu_seconds_total[1m]) * 100')
        ram     = self._query_prometheus('process_resident_memory_bytes / 1024 / 1024')
        latency = self._query_prometheus('histogram_quantile(0.95, rate(service_request_latency_seconds_bucket[5m]))')
        drift   = self._query_prometheus('inference_drift_score')
        
        metrics = torch.tensor(
            list(zip(cpu, ram, latency, drift)),
            dtype=torch.float32
        ).unsqueeze(0).to(self.device)
        
        data = (metrics - self.mean.to(self.device)) / (self.std.to(self.device) + 1e-8)
        with torch.no_grad():
            pred = self.model(data)
        
        # Denormalize
        pred = pred * self.std.to(self.device) + self.mean.to(self.device)
        pred = pred.cpu().numpy()
        
        if not np.isfinite(pred).all():
            logger.warning("LSTM predicted NaN/inf — returning zeros")
            pred = np.zeros_like(pred)
        
        return {
            "predicted_cpu":     pred[0, :, 0].tolist(),
            "predicted_ram":     pred[0, :, 1].tolist(),
            "predicted_latency": pred[0, :, 2].tolist(),
            "predicted_drift":   pred[0, :, 3].tolist(),
            "steps":             self.output_steps
        }

In [ ]:
try:
    config = ConfigManager()
    predictor = LSTM_Predictor_(config=config.get_lstm_config())
    predictor.predict()
except Exception as e:
    raise CustomException(e, sys)